# Clase 14 — Model as a Service y Streamlit

**Módulo:** Del modelo al producto: diseño y trazabilidad MLOps

**Intención:** convertir el modelo seleccionado por el pipeline en un servicio local con un contrato observable y conectarlo con una interfaz que actúe como cliente.

> **Quiz 4 — Clases 11, 12 y 13**  
> Código de acceso: `registryflujo2026`

El quiz se realiza al inicio de la clase y su disponibilidad se controla desde Canvas.

## Antes de comenzar: actualiza el repositorio

Desde la raíz del repositorio público revisa primero que no tengas cambios propios. Después actualiza `main` y sincroniza el ambiente bloqueado:

```bash
git status --short
git switch main
git pull
uv sync --locked
```

Si `git status --short` muestra archivos modificados, detente antes de cambiar de rama o actualizar. Esos cambios necesitan revisarse para no perder trabajo.

## Punto de partida

**Prerrequisitos concretos:** reconocer un modelo registrado, una versión y un alias; explicar que el `Pipeline` mantiene juntos preprocesamiento y estimador; localizar el modelo de la Clase 13 en Unity Catalog; y enviar un cuerpo JSON a un endpoint `POST`.

**Resultado observable:** una solicitud válida atraviesa Streamlit y una API FastAPI, llega al `Pipeline` registrado bajo `champion` y regresa con una duración estimada, el alias y la versión utilizada.

El entrenamiento ya terminó. Esta clase no vuelve a descargar los históricos, ejecutar HPO ni ajustar un modelo por solicitud.

## 1. Model as a Service

Un servicio es una abstracción utilizada por otras aplicaciones. Expone una interfaz que puede llamarse de forma estable y mantiene ocultos los detalles de implementación. El consumidor necesita conocer la URL, el método, la forma de la solicitud y la forma de la respuesta; no necesita conocer el lenguaje o las bibliotecas con las que se entrenó el modelo.

Para un modelo servido mediante una API REST, una aplicación envía datos por HTTP y recibe una respuesta JSON. La documentación del contrato permite saber qué campos y tipos acepta el servicio. FastAPI genera esa documentación mediante OpenAPI.

![Una aplicación web envía datos a un servicio independiente que contiene el modelo y recibe una predicción](../assets/modulo-02-ciclo-mlops/clase-09/model-as-a-service.png)

*El patrón separa la aplicación cliente del servicio que ejecuta el modelo. La imagen histórica se conserva sin modificaciones. Fuente: INNOQ, [Three Levels of ML Software](https://ml-ops.org/content/three-levels-of-ml-software), CC BY 4.0.*

## 2. El modelo seleccionado por el pipeline

La Clase 13 registró tres modelos finales bajo un nombre de tres niveles en Unity Catalog y asignó aliases según el RMSE sobre abril:

```text
menor RMSE → champion
segundo    → challenger
tercero    → candidate
```

El nombre esperado tiene esta forma:

```text
workspace.default.nyc_taxi_trip_duration_clase_13
```

La URI agrega el alias elegido:

```text
models:/workspace.default.nyc_taxi_trip_duration_clase_13@champion
```

Una URI identifica qué recurso cargar. El alias es mutable: puede apuntar después a otra versión, mientras cada versión permanece inmutable.

### 2.1 Un solo contrato de entrada

El modelo conserva las cinco features utilizadas desde la Clase 6:

| Campo | Tipo y regla del contrato del curso |
|---|---|
| `distancia_km` | número mayor que 0 y hasta 100 |
| `pasajeros` | entero entre 1 y 6 |
| `hora_recoleccion` | entero entre 0 y 23 |
| `zona_origen` | entero entre 1 y 265 |
| `zona_destino` | entero entre 1 y 265 |

Las zonas siguen siendo categorías aunque se representen con enteros. El `Pipeline` registrado aplica `OneHotEncoder(handle_unknown="ignore")` y conserva las variables numéricas sin repetir esa lógica en el servicio.

## 3. Servir localmente un MLflow Model

`mlflow models serve` descarga el MLflow Model indicado por la URI, carga su ambiente y levanta un servidor de inferencia local. El comando evita escribir una API propia cuando basta un contrato estándar para probar el artefacto.

El servidor ofrece estos endpoints:

| Endpoint | Propósito |
|---|---|
| `POST /invocations` | recibir entradas y devolver predicciones |
| `GET /ping` y `GET /health` | confirmar que el proceso responde |
| `GET /version` | consultar la versión de MLflow del servidor |

MLflow implementa este servidor con FastAPI. Por eso no compararemos dos frameworks: compararemos una interfaz estándar generada por MLflow con una API específica que escribiremos para Green Taxi.

### 3.1 Preparar la configuración externa

El archivo `.env` de la raíz ya contiene `DATABRICKS_HOST` y `DATABRICKS_TOKEN` desde las clases anteriores. Mantén esos valores fuera de Git y agrega estas líneas:

```dotenv
MLFLOW_TRACKING_URI=databricks
MLFLOW_REGISTRY_URI=databricks-uc
MODELO_REGISTRADO=workspace.default.nyc_taxi_trip_duration_clase_13
ALIAS_MODELO=champion
API_URL=http://127.0.0.1:8000/api/v1/predicciones
```

`workspace` es el catálogo disponible en Databricks Free Edition para esta práctica y `default` es el esquema usado por el pipeline. `MODELO_REGISTRADO` conserva el nombre de tres niveles, sin el prefijo `models:/` y sin el alias. Nunca copies el token al notebook, a un comando compartido ni a un commit.

### 3.2 Levantar el servidor estándar

En una terminal ubicada en la raíz ejecuta el servidor con la URI completa del modelo. Hay dos opciones relacionadas con el ambiente, pero cumplen funciones diferentes:

- `--env-file .env` entrega al proceso `DATABRICKS_HOST`, `DATABRICKS_TOKEN` y las URI de MLflow. Es necesario para acceder desde la computadora local al modelo de Unity Catalog.
- `--env-manager local` es opcional. Indica a MLflow que reutilice el ambiente ya creado por `uv sync --locked`. Si se omite, MLflow usa por defecto `virtualenv` e intenta crear un ambiente aislado a partir de las dependencias registradas con el modelo. Ese aislamiento puede ser útil al desplegar un artefacto en otro contexto; en esta práctica usamos `local` para evitar una segunda instalación, reducir el tiempo de arranque y mantener las mismas versiones que acabamos de sincronizar con `uv`.

`-p 5001` asigna el puerto del servidor y evita competir con el puerto histórico del Tracking Server.

```bash
uv run --env-file .env mlflow models serve \
  -m "models:/workspace.default.nyc_taxi_trip_duration_clase_13@champion" \
  -p 5001 \
  --env-manager local
```

Mantén esa terminal abierta. El proceso ocupa la terminal porque queda escuchando solicitudes.

### 3.3 Probar el servidor estándar con Postman

Crea una solicitud nueva en Postman con estas opciones:

1. método `POST`;
2. URL `http://127.0.0.1:5001/invocations`;
3. pestaña **Body** → **raw** → **JSON**.

`dataframe_split` separa los nombres de columnas de las filas y conserva explícitamente el orden de las cinco features. Copia y pega el bloque completo en el body:

```json
{
  "dataframe_split": {
    "columns": [
      "distancia_km",
      "pasajeros",
      "hora_recoleccion",
      "zona_origen",
      "zona_destino"
    ],
    "data": [[4.2, 2, 18, 75, 42]]
  }
}
```

Postman agrega `Content-Type: application/json` al elegir JSON. Presiona **Send** y comprueba que la respuesta contenga `predictions`. Para observar los otros endpoints, crea dos solicitudes `GET`: una a `http://127.0.0.1:5001/health` y otra a `http://127.0.0.1:5001/version`.

El body de MLflow es deliberadamente genérico: el mismo endpoint debe poder recibir datos para modelos distintos.

### ✅ Checkpoint: modelo y contrato estándar

Antes de continuar, confirma que puedes relacionar cuatro elementos: nombre registrado, alias `champion`, versión concreta y respuesta de `/invocations`. Si el servidor no inicia, revisa primero el nombre de tres niveles, las variables del `.env`, los permisos sobre el modelo y que el puerto 5001 esté libre.

## 4. Dos enfoques para servir el mismo modelo

Un **contrato HTTP** es el acuerdo entre cliente y servidor: ruta, método, cuerpo esperado y forma de la respuesta. Aquí veremos dos enfoques válidos para exponer el mismo MLflow Model. La elección depende de cuánto control necesita la aplicación.

| `mlflow models serve` | API escrita con FastAPI |
|---|---|
| Genera el servidor sin escribir endpoints. | Requiere escribir y mantener la aplicación. |
| Usa `/invocations` y formatos genéricos como `dataframe_split`. | Permite elegir `/api/v1/predicciones` y recibir un JSON directo. |
| Es conveniente para probar un MLflow Model o conectar clientes que conocen el protocolo de MLflow. | Es conveniente cuando una aplicación necesita nombres, validaciones, respuestas o reglas propias. |
| Devuelve una estructura estándar de predicciones. | Puede devolver duración, modelo, alias y versión. |
| Reduce código y decisiones de interfaz. | Ofrece más control y también más responsabilidad. |

Los dos enfoques cargan el mismo artefacto. FastAPI no reconstruye el modelo: define una interfaz más específica para el caso Green Taxi. MLflow Model Serving ya usa FastAPI internamente, pero genera esa aplicación por nosotros.

## 5. Preparar el trabajo local

La práctica no es una entrega y no requiere rama, commit ni PR. Desde la raíz crea la carpeta ignorada y confirma tu ubicación:

```bash
mkdir -p labs/trabajo-local/clase-14
cd labs/trabajo-local/clase-14
pwd
```

Regresa a la raíz antes de ejecutar los comandos posteriores:

```bash
cd ../../..
```

Crearás desde cero `api.py` y `streamlit_app.py`. Su estructura forma parte del aprendizaje, por lo que no existe un starter para esos dos archivos. El starter de depuración de la sección 10 es una actividad independiente y no los sustituye.

## 6. Construir la API FastAPI

Crea `labs/trabajo-local/clase-14/api.py`:

```python
import os

import mlflow
import pandas as pd
from dotenv import load_dotenv
from fastapi import FastAPI
from mlflow import MlflowClient
from pydantic import BaseModel, Field

load_dotenv(override=True)

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "databricks")
REGISTRY_URI = os.getenv("MLFLOW_REGISTRY_URI", "databricks-uc")
MODELO_REGISTRADO = os.environ["MODELO_REGISTRADO"]
ALIAS_MODELO = os.getenv("ALIAS_MODELO", "champion")
MODELO_URI = f"models:/{MODELO_REGISTRADO}@{ALIAS_MODELO}"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(REGISTRY_URI)

cliente = MlflowClient(registry_uri=REGISTRY_URI)
version = cliente.get_model_version_by_alias(MODELO_REGISTRADO, ALIAS_MODELO)
modelo = mlflow.pyfunc.load_model(MODELO_URI)


class SolicitudPrediccion(BaseModel):
    distancia_km: float = Field(gt=0, le=100)
    pasajeros: int = Field(ge=1, le=6)
    hora_recoleccion: int = Field(ge=0, le=23)
    zona_origen: int = Field(ge=1, le=265)
    zona_destino: int = Field(ge=1, le=265)


class RespuestaPrediccion(BaseModel):
    duracion_estimada_minutos: float
    modelo: str
    alias: str
    version: str


app = FastAPI(title="API de duración de viajes Green Taxi")


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}


@app.post("/api/v1/predicciones", response_model=RespuestaPrediccion)
def crear_prediccion(solicitud: SolicitudPrediccion) -> RespuestaPrediccion:
    entrada = pd.DataFrame([solicitud.model_dump()])
    duracion = float(modelo.predict(entrada)[0])
    return RespuestaPrediccion(
        duracion_estimada_minutos=round(duracion, 1),
        modelo=MODELO_REGISTRADO,
        alias=ALIAS_MODELO,
        version=str(version.version),
    )
```

### 6.1 Qué responsabilidad tiene cada bloque

1. `load_dotenv` obtiene configuración y autenticación fuera del código.
2. Las URI indican dónde están Tracking, Registry y el modelo seleccionado.
3. `get_model_version_by_alias` resuelve qué versión concreta ocupa `champion`.
4. `load_model` carga una vez el `Pipeline` completo al iniciar el proceso.
5. Pydantic valida la solicitud antes de llamar al modelo.
6. El endpoint convierte un viaje en un DataFrame de una fila y devuelve una respuesta específica para Green Taxi.

La metadata identifica qué se utilizó para responder. Si el alias cambia mientras la API está encendida, el proceso debe reiniciarse para cargar la nueva versión.

### 6.2 Levantar y explorar la API

Abre una terminal nueva en la raíz. `--port 8000` asigna un puerto diferente al servidor estándar de MLflow:

```bash
uv run --env-file .env fastapi dev labs/trabajo-local/clase-14/api.py --port 8000
```

Mantén el proceso abierto y visita `http://127.0.0.1:8000/docs`. Comprueba también `http://127.0.0.1:8000/health` en el navegador o mediante una solicitud `GET` en Postman.

### 6.3 Probar la API propia con Postman

Crea otra solicitud en Postman: método `POST`, URL `http://127.0.0.1:8000/api/v1/predicciones` y **Body** → **raw** → **JSON**. Esta API ya no requiere `dataframe_split`; usa directamente el vocabulario del caso. Copia y pega el payload:

```json
{
  "distancia_km": 4.2,
  "pasajeros": 2,
  "hora_recoleccion": 18,
  "zona_origen": 75,
  "zona_destino": 42
}
```

La respuesta debe incluir `duracion_estimada_minutos`, `modelo`, `alias` y `version`. Después cambia `hora_recoleccion` a `24`: Pydantic debe responder `422` antes de invocar el modelo.

### ✅ Checkpoint: comparar los dos enfoques

Compara en Postman la solicitud enviada a `/invocations` con la enviada a `/api/v1/predicciones`. Ambas llegan al mismo modelo. La primera usa el formato estándar de MLflow; la segunda usa una interfaz específica para Green Taxi y agrega metadata que permite identificar la versión.

## 7. Streamlit como consumidor

Streamlit permite construir una interfaz web desde Python mediante componentes como campos numéricos, botones y mensajes. En este sistema es un **cliente**: recopila los datos del viaje y llama a FastAPI mediante HTTP. No carga el modelo ni ejecuta `predict` directamente.

![Una persona captura datos en Streamlit; Streamlit envía JSON a FastAPI; FastAPI usa el MLflow Model champion y devuelve la predicción](../assets/modulo-02-ciclo-mlops/clase-14/flujo-streamlit-fastapi-mlflow.svg)

*La solicitud avanza de izquierda a derecha: datos del viaje → JSON → DataFrame → modelo. La respuesta regresa con duración, alias y versión. Cada bloque tiene una sola responsabilidad.*

### 7.1 Primera versión: formulario y payload visible

Construiremos la interfaz en dos pasos. Primero comprobaremos que los componentes y los tipos producen el diccionario esperado, todavía sin llamar a la API. Crea `labs/trabajo-local/clase-14/streamlit_app.py`:

```python
import streamlit as st

st.set_page_config(page_title="Duración de viajes Green Taxi")
st.title("Duración estimada de un viaje Green Taxi")
st.write("Captura las cinco características disponibles antes del viaje.")

with st.sidebar:
    st.header("Datos del viaje")
    distancia_km = st.number_input(
        "Distancia aproximada (km)",
        min_value=0.1,
        max_value=100.0,
        value=4.2,
        step=0.1,
        format="%.1f",
        help="Distancia disponible antes de iniciar el viaje.",
    )
    pasajeros = st.number_input("Número de pasajeros", min_value=1, max_value=6, value=2, step=1)
    hora_recoleccion = st.number_input(
        "Hora de recolección", min_value=0, max_value=23, value=18, step=1
    )
    zona_origen = st.number_input("Zona de origen", min_value=1, max_value=265, value=75, step=1)
    zona_destino = st.number_input("Zona de destino", min_value=1, max_value=265, value=42, step=1)

entrada = {
    "distancia_km": float(distancia_km),
    "pasajeros": int(pasajeros),
    "hora_recoleccion": int(hora_recoleccion),
    "zona_origen": int(zona_origen),
    "zona_destino": int(zona_destino),
}

st.subheader("Payload que enviará la interfaz")
st.json(entrada)
```

### 7.2 Ejecutar la primera versión

Desde la raíz ejecuta:

```bash
uv run --env-file .env streamlit run \
  labs/trabajo-local/clase-14/streamlit_app.py \
  --server.port 8501
```

El comando se lee así:

- `uv run` usa el ambiente reproducible del proyecto;
- `--env-file .env` entrega la configuración externa al proceso; en la segunda versión permitirá leer `API_URL`;
- `streamlit run` inicia el servidor de desarrollo de Streamlit;
- la ruta señala el archivo que Streamlit debe ejecutar;
- `--server.port 8501` fija el puerto para que la URL no cambie durante la práctica.

Streamlit ejecuta el archivo completo de arriba abajo cada vez que cambia un widget. Modifica una entrada y confirma que el JSON visible cambia con ella. Mantén esta terminal abierta.

### 7.3 Segunda versión: llamar a FastAPI

Ahora conecta la interfaz con la API. Agrega al inicio del archivo las importaciones y configuración siguientes:

```python
import os

import httpx
import streamlit as st
from dotenv import load_dotenv

load_dotenv(override=True)

API_URL = os.getenv(
    "API_URL",
    "http://127.0.0.1:8000/api/v1/predicciones",
)
```

Conserva una sola línea `import streamlit as st`. Después reemplaza `st.subheader(...)` y `st.json(entrada)` del final por este bloque:

```python
if st.button("Estimar duración", type="primary"):
    try:
        respuesta = httpx.post(API_URL, json=entrada, timeout=10.0)
        respuesta.raise_for_status()
    except httpx.HTTPStatusError as error:
        st.error(f"La API rechazó la solicitud: {error.response.text}")
    except httpx.RequestError:
        st.error("No fue posible conectarse con la API de predicción.")
    else:
        resultado = respuesta.json()
        st.metric(
            "Duración estimada",
            f"{resultado['duracion_estimada_minutos']:.1f} min",
        )
        st.caption(
            f"Modelo: {resultado['modelo']} · alias: {resultado['alias']} · "
            f"versión: {resultado['version']}"
        )
```

`httpx.post(..., json=entrada)` serializa el diccionario y envía JSON. `timeout=10.0` evita esperar indefinidamente. `HTTPStatusError` significa que la API respondió con un estado no exitoso; `RequestError` significa que no pudo completarse la conexión.

Streamlit detectará el archivo guardado y volverá a ejecutarlo. Con FastAPI todavía activo, conserva los valores de ejemplo y presiona **Estimar duración**. La pantalla debe mostrar duración, nombre registrado, `champion` y versión.

### ✅ Checkpoint: recorrido completo

Confirma que puedes seguir una solicitud sin saltos:

1. Streamlit reúne las cinco features.
2. `httpx` envía JSON a `/api/v1/predicciones`.
3. Pydantic valida campos y rangos.
4. FastAPI construye un DataFrame de una fila.
5. El `Pipeline` registrado transforma y predice.
6. La respuesta identifica alias y versión.
7. Streamlit presenta el resultado.

## 8. Contingencia si Databricks no está disponible

Un problema de autenticación, cuota o conectividad no debe consumir toda la sesión. La alternativa docente usa el Model Registry local de las Clases 10–11, que conserva las mismas cinco features y un `Pipeline` completo. En ese caso se ajustan en `.env` las URI y el nombre del modelo local, y se mantiene igual el contrato FastAPI–Streamlit.

La contingencia cambia dónde se obtiene el artefacto; no cambia la arquitectura, las entradas ni las responsabilidades de las capas. No copies tokens ni los compartas para resolver una falla de acceso.

## 9. Diagnóstico de errores frecuentes

| Síntoma | Causa probable | Primera verificación |
|---|---|---|
| El modelo no se descarga. | Catálogo, nombre, alias, token o permisos incorrectos. | Copiar el nombre de tres niveles desde Catalog Explorer y revisar sólo la presencia de las variables. |
| El puerto ya está ocupado. | Otro servidor continúa ejecutándose. | Identificar cuál terminal mantiene el proceso y detenerlo con `Ctrl+C`. |
| MLflow rechaza `/invocations`. | Formato o columnas incompatibles. | Comparar `columns` y `data` con la firma registrada. |
| FastAPI devuelve `422`. | Falta un campo, el tipo no coincide o el valor está fuera de rango. | Leer `detail` y revisar el JSON en `/docs`. |
| FastAPI devuelve `404`. | La ruta del cliente no coincide con el decorador. | Comparar exactamente `/api/v1/predicciones`. |
| Streamlit no se conecta. | La API está detenida o `API_URL` apunta a otro puerto. | Consultar `/health` directamente. |
| La interfaz muestra campos antiguos. | Se reutilizó el contrato histórico de tres variables. | Confirmar las cinco features vigentes. |
| La versión no cambia al reasignar el alias. | La API cargó el modelo al iniciar. | Reiniciar FastAPI después de un cambio aprobado de alias. |

## 10. Actividad: depurar una API con cinco errores

En esta actividad trabajarás con una API local que contiene exactamente cinco errores intencionales. La estimación es una función determinista incluida en el archivo: no usa Databricks, credenciales, red ni el modelo registrado. Así, cada síntoma procede del código que puedes observar.

> **Política de herramientas:** durante esta actividad no utilices ChatGPT, Codex, Copilot, agentes de IA ni asistentes generativos. Puedes consultar el código, la terminal, los tracebacks, los logs, Postman, `/docs`, este notebook y documentación oficial.

### 10.1 Preparar copias editables

Detén con `Ctrl+C` la API FastAPI construida anteriormente para liberar el puerto 8000. Desde la raíz, copia los dos archivos del starter a la carpeta local que ya creaste:

```bash
cp labs/starters/clase-14-depuracion-api/api_con_errores.py \
  labs/starters/clase-14-depuracion-api/hallazgos.md \
  labs/trabajo-local/clase-14/
pwd
ls labs/trabajo-local/clase-14
```

Edita únicamente las copias dentro de `labs/trabajo-local/clase-14/`. El starter rastreado debe permanecer intacto para que futuros `git pull` no entren en conflicto. El checkpoint correcto muestra `api_con_errores.py` y `hallazgos.md` junto con los archivos creados anteriormente.

### 10.2 Ejecutar, observar y corregir

Inicia la API desde la raíz:

```bash
uv run fastapi run \
  labs/trabajo-local/clase-14/api_con_errores.py \
  --port 8000
```

`fastapi run` inicia la aplicación sin recarga automática. Esto obliga a detenerla y volver a ejecutar el mismo comando después de cada cambio. Si el proceso termina por un error de arranque, lee primero el último bloque del traceback.

Para cada error sigue el mismo ciclo:

1. ejecuta la API o envía una solicitud;
2. observa el traceback, log, código HTTP o JSON;
3. registra la evidencia y una hipótesis en `hallazgos.md`;
4. realiza **un solo cambio**;
5. detén el proceso con `Ctrl+C` si continúa activo;
6. vuelve a ejecutar la API y registra el nuevo resultado.

No corrijas problemas que todavía no hayas observado. Un cambio acertado debe producir nueva evidencia o hacer que una verificación pase.

### 10.3 Verificar el comportamiento

Cuando el servidor inicie, usa el navegador o Postman para comprobar `GET http://127.0.0.1:8000/health`. Después envía desde Postman una solicitud `POST` a `http://127.0.0.1:8000/api/v1/predicciones` con este payload válido:

```json
{
  "distancia_km": 4.2,
  "pasajeros": 2,
  "hora_recoleccion": 18,
  "zona_origen": 75,
  "zona_destino": 42
}
```

Una respuesta `200` para el caso válido no demuestra todavía que la validación sea correcta. Repite la solicitud con `"hora_recoleccion": 24`.

La API está corregida sólo cuando cumple todas estas condiciones:

- inicia sin traceback;
- `/health` responde `200` con `{"status": "ok"}`;
- el payload válido responde `200`;
- la respuesta contiene `duracion_estimada_minutos`;
- `hora_recoleccion=24` responde `422`;
- `hallazgos.md` documenta cinco cambios con evidencia.

### ✅ Checkpoint: explicar un diagnóstico

Elige uno de los errores y explica qué observaste, qué evidencia fue útil, qué hipótesis formulaste, cuál fue el cambio mínimo y cómo comprobaste la corrección. `hallazgos.md` es trabajo local: no requiere rama, commit, pull request ni entrega en Canvas.

## Recopilación

- Model as a Service separa el consumidor de los detalles internos del modelo.
- El alias identifica un rol mutable; la versión identifica un artefacto inmutable.
- El MLflow Model de Clase 13 ya contiene preprocesamiento y estimador.
- MLflow ofrece un servidor local y un contrato de inferencia estándar.
- La API FastAPI propia adapta ese modelo al vocabulario y validaciones de Green Taxi.
- Streamlit consume la API mediante HTTP; no carga el modelo directamente.
- Los errores se diagnostican con un ciclo de ejecución, evidencia, hipótesis, cambio mínimo y nueva verificación.

## ✅ Check final de la clase

Antes de cerrar, comprueba que puedes:

- obtener una predicción válida desde Streamlit;
- identificar el endpoint y el JSON enviados a FastAPI;
- localizar en la respuesta el alias y la versión usados;
- explicar por qué no existe un preprocesador separado;
- distinguir la interfaz estándar de MLflow de la API específica de Green Taxi;
- describir qué proceso debe estar activo en cada terminal;
- justificar una corrección a partir de un traceback, log, código HTTP o respuesta JSON.

Este check detecta dudas; no es una entrega en Canvas.

## Referencias

- [MLflow — Deploy MLflow Model as a Local Inference Server](https://mlflow.org/docs/latest/ml/deployment/deploy-model-locally/)
- [MLflow — Model Registry Workflows](https://mlflow.org/docs/latest/ml/model-registry/workflow/)
- [FastAPI — Request Body](https://fastapi.tiangolo.com/tutorial/body/)
- [Streamlit — `st.number_input`](https://docs.streamlit.io/develop/api-reference/widgets/st.number_input)
- [HTTPX — QuickStart](https://www.python-httpx.org/quickstart/)
- [INNOQ — Three Levels of ML Software](https://ml-ops.org/content/three-levels-of-ml-software)